# 6G7V0017 Advanced Machine Learning — 1CWK100

**Student:** Sajan Mathew | **ID:** 25928716 | **Module:** 6G7V0017 AML
**Dataset:** AutoTrader Car Sale Adverts | **Task:** Supervised Regression — predict `price`

> Run all cells top to bottom in Google Colab. Upload `adverts.csv` when prompted.

## 0. Setup — Installs, Imports, Constants

In [ ]:
!pip install xgboost shap --quiet

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

from pathlib import Path
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import (
    RandomForestRegressor, ExtraTreesRegressor,
    HistGradientBoostingRegressor,
    VotingRegressor, StackingRegressor
)
from sklearn.feature_selection import (
    SelectKBest, f_regression, SelectFromModel,
    RFE, SequentialFeatureSelector
)
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE, Isomap
from sklearn.cluster import KMeans
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error,
    r2_score, make_scorer, silhouette_score
)
from sklearn.inspection import permutation_importance, PartialDependenceDisplay

try:
    from xgboost import XGBRegressor
    HAS_XGB = True; print('XGBoost loaded')
except ImportError:
    HAS_XGB = False; print('XGBoost not available')

try:
    import shap
    HAS_SHAP = True; print('SHAP loaded')
except ImportError:
    HAS_SHAP = False; print('SHAP not available — run: !pip install shap')

SEED = 42
np.random.seed(SEED)
OUTPUT_DIR = Path('aml_outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

# Two separate constants — different purposes
VALID_REG_YEAR_MIN = 1900   # validity window for cleaning
VALID_REG_YEAR_MAX = 2020   # dataset max year_of_registration
REFERENCE_YEAR     = 2020   # used as the anchor for vehicle_age
SAMPLE_SIZE        = 75_000
## ~UK average miles/year - usage-intensity proxy benchmark only.
## Treated as an approximate reference rather than a precise official valuation rule.
UK_AVG_MILES_YEAR  = 7100

print('All imports successful.')

## 0.1 Load Dataset

In [ ]:
## Upload adverts.csv when prompted
## Or mount Google Drive:
##   from google.colab import drive; drive.mount('/content/drive')
##   DATA_PATH = '/content/drive/MyDrive/adverts.csv'

from google.colab import files
print('Upload adverts.csv when prompted...')
uploaded = files.upload()
DATA_PATH = list(uploaded.keys())[0]

raw_df = pd.read_csv(DATA_PATH)
print(f'Raw dataset: {raw_df.shape[0]:,} rows x {raw_df.shape[1]} columns')
raw_df.head(3)

---
# 1. Data Description and Pre-Processing (10%)

**Market Heuristics approach:** Feature engineering is guided by the perspective of an AutoTrader marketplace analyst. Each derived feature answers a question that a real buyer, seller, or platform analyst would ask. This shifts the model from a generic price predictor toward a risk-adjusted transaction facilitator grounded in UK automotive market reality.

The business target is advertised vehicle price in GBP. For modelling, `log1p(price)` is used to reduce right-skew. All metrics are reported in original £ via `np.expm1`.

In [ ]:
## 1.1 Data Audit
data_audit = pd.DataFrame({
    'dtype':       raw_df.dtypes.astype(str),
    'missing':     raw_df.isna().sum(),
    'missing_pct': (raw_df.isna().mean()*100).round(2),
    'unique':      raw_df.nunique(dropna=False),
})
print(f'Shape: {raw_df.shape[0]:,} rows x {raw_df.shape[1]} columns')
display(data_audit)
data_audit.to_csv(OUTPUT_DIR / '01_data_audit.csv')

In [ ]:
## 1.2 Cleaning Pipeline
## Leakage-free: no fitted state — pure column-wise transforms

def uk_reg_code_to_year(code):
    """Recover year_of_registration from DVLA reg_code suffix.
    UK plate codes encode registration year:
      02-24 = March plates 2002-2024 | 51-74 = September plates 2001-2024"""
    try:
        num = int(str(code).strip()[:2])
        if 2 <= num <= 24:  return 2000 + num
        if 51 <= num <= 74: return 1950 + num
    except (ValueError, TypeError): pass
    return np.nan

df = raw_df.copy(deep=True)

## Extract advert date from public_reference prefix
df['advert_date'] = pd.to_datetime(
    df['public_reference'].astype(str).str[:8], format='%Y%m%d', errors='coerce')
df['advert_year'] = df['advert_date'].dt.year

## Nullify impossible registration years
df['year_of_registration'] = pd.to_numeric(df['year_of_registration'], errors='coerce')
bad = df['year_of_registration'].notna() & ~df['year_of_registration'].between(
    VALID_REG_YEAR_MIN, VALID_REG_YEAR_MAX)
df.loc[bad, 'year_of_registration'] = np.nan
print(f'Missing year before reg_code recovery: {df["year_of_registration"].isna().sum():,}')

## Use reg_code to recover missing year_of_registration
## year_was_imputed flag preserves the missingness signal: rows where the year had to
## be recovered may still behave differently from rows with a clean original year.
pre_recovery_missing = df['year_of_registration'].isna()
mask = pre_recovery_missing & df['reg_code'].notna()
df.loc[mask, 'year_of_registration'] = df.loc[mask, 'reg_code'].apply(uk_reg_code_to_year)
df['year_was_imputed'] = (pre_recovery_missing & df['year_of_registration'].notna()).astype(int)
print(f'Missing year after reg_code recovery:  {df["year_of_registration"].isna().sum():,}')
print(f'Rows with year_was_imputed flag set:   {df["year_was_imputed"].sum():,}')

## Remove explicit price placeholders and extreme mileage noise
remove = (df['price'] == 9_999_999) | (df['mileage'].fillna(-1) >= 900_000)
df = df[~remove].copy()

## Hard price guard: drop adverts outside [£200, £250,000]
## Sub-£200 are scrap/parts listings that distort log_price; >£250k are luxury
## outliers we cannot model with this feature set anyway.
df = df[df['price'].between(200, 250_000)].copy()

## Negative vehicle age → set registration year to NaN
prelim_age = df['advert_year'] - df['year_of_registration']
df.loc[prelim_age < 0, 'year_of_registration'] = np.nan
df.loc[prelim_age > 100, 'year_of_registration'] = np.nan

## Fill categorical missing with Unknown
for col in ['standard_colour','standard_make','standard_model',
            'vehicle_condition','body_type','fuel_type']:
    df[col] = df[col].fillna('Unknown')

cleaned_df = df.copy()
print(f'Cleaned dataset: {len(cleaned_df):,} rows')
print(f'Rows removed: {len(raw_df) - len(cleaned_df):,}')

In [ ]:
## 1.3 Feature Engineering — Market Heuristics
## Each feature answers a question a real buyer or seller would ask

## vehicle_age: primary depreciation driver
## Anchored at REFERENCE_YEAR (dataset max year_of_registration) rather than per-advert
## year, so the feature is stable across adverts and not noisy with advert timing.
cleaned_df['vehicle_age'] = REFERENCE_YEAR - cleaned_df['year_of_registration']

## mileage_per_year: usage intensity (total mileage alone is incomplete)
## A 2-year-old car with 50,000 miles differs from a 10-year-old with 50,000 miles
cleaned_df['mileage_per_year'] = np.where(
    cleaned_df['vehicle_age'] > 0,
    cleaned_df['mileage'] / cleaned_df['vehicle_age'],
    cleaned_df['mileage']
)

## age_adjusted_mileage_score: has this car been driven harder than the UK average?
## Score < 1.0 = babied | Score > 1.0 = driven hard
## Reference benchmark only - UK_AVG_MILES_YEAR is a rough usage-intensity proxy.
## NOTE: vehicle_age == 0 produces a degenerate denominator under naive division.
## We treat new cars as 1 reference-year of expected mileage so a brand-new car
## with 300 miles cannot blow up to an absurd score, and we clip the result so
## this single feature cannot dominate the model as an artificial outlier.
cleaned_df['age_adjusted_mileage'] = np.where(
    cleaned_df['vehicle_age'] > 0,
    cleaned_df['mileage'] / (cleaned_df['vehicle_age'] * UK_AVG_MILES_YEAR),
    cleaned_df['mileage'] / UK_AVG_MILES_YEAR
)
cleaned_df['age_adjusted_mileage'] = cleaned_df['age_adjusted_mileage'].clip(0, 10)

## depreciation_stage: where is this car in its depreciation lifecycle?
## Encodes the non-linear depreciation curve directly rather than treating age as linear
def depreciation_stage(age):
    if pd.isna(age):          return 'unknown'
    if age == 0:              return 'new'
    elif age <= 1:            return 'cliff'      ## largest single-year drop
    elif age <= 3:            return 'steep'      ## rapid depreciation
    elif age <= 7:            return 'moderate'   ## slowing
    elif age <= 12:           return 'slow'       ## near-stable
    else:                     return 'floor'      ## scrap value territory
cleaned_df['depreciation_stage'] = cleaned_df['vehicle_age'].apply(depreciation_stage)

## is_sweet_spot: AutoTrader's highest-demand segment
## 2-5 years old, under 40,000 miles, USED condition
cleaned_df['is_sweet_spot'] = (
    (cleaned_df['vehicle_age'].between(2, 5)) &
    (cleaned_df['mileage'] < 40_000) &
    (cleaned_df['vehicle_condition'] == 'USED')
).astype(int)

## mileage_is_round: listing trustworthiness signal
## Real odometers produce 47,382 not 45,000 — round numbers are estimates
cleaned_df['mileage_is_round'] = (cleaned_df['mileage'] % 5000 == 0).astype(int)

## colour_premium: colour tax — every buyer knows this, no notebook encodes it
PREMIUM_COLOURS = {'Black', 'White', 'Silver', 'Grey'}
DISCOUNT_COLOURS = {'Green', 'Yellow', 'Orange', 'Purple', 'Pink'}
def colour_premium(c):
    if c in PREMIUM_COLOURS:  return 1
    if c in DISCOUNT_COLOURS: return -1
    return 0
cleaned_df['colour_premium'] = cleaned_df['standard_colour'].map(colour_premium)

## is_plate_change_month: UK plates change March/September — real market timing effect
cleaned_df['advert_month'] = cleaned_df['advert_date'].dt.month
cleaned_df['is_plate_change_month'] = cleaned_df['advert_month'].isin([3, 9]).astype(int)

## implicit_repair_risk: proxy for MOT anxiety — old + high mileage = upcoming costs
cleaned_df['implicit_repair_risk'] = (
    (cleaned_df['vehicle_age'] > 8) & (cleaned_df['mileage'] > 80_000)
).astype(int)

## is_alternative_fuel: electric/hybrid technology carries distinct pricing signal
cleaned_df['is_alternative_fuel'] = cleaned_df['fuel_type'].isin(
    ['Electric', 'Hybrid', 'Plug-in Hybrid']).astype(int)

## log_price: target transformation — reduces right skew for modelling
## price_per_mile was considered but REJECTED because it uses the target variable
## and would create target leakage
cleaned_df['log_price'] = np.log1p(cleaned_df['price'])

## Cleaning + feature-engineering sanity checks
## Fail loudly here if data does not match the claims we will make in the report.
assert (cleaned_df['price'] > 0).all(), 'Non-positive prices remain.'
assert cleaned_df['vehicle_age'].dropna().between(0, 100).all(), 'Invalid vehicle ages remain.'
assert (cleaned_df['mileage'].dropna() < 900_000).all(), 'Extreme mileage remains.'
assert cleaned_df['log_price'].notna().all(), 'Missing log_price values remain.'
print('Cleaning assertions passed.')

engineered = ['vehicle_age','mileage_per_year','age_adjusted_mileage',
              'depreciation_stage','is_sweet_spot','mileage_is_round',
              'colour_premium','is_plate_change_month','implicit_repair_risk',
              'is_alternative_fuel']
print(f'Engineered Market Heuristics: {len(engineered)} features')
print(cleaned_df[engineered].head(3))

In [ ]:
## 1.4 EDA Plots and Distributions

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

## Price distribution before/after log
axes[0,0].hist(cleaned_df['price'], bins=80, color='#2E86AB', edgecolor='white')
axes[0,0].set_title(f'Price (raw) skew={cleaned_df["price"].skew():.1f}')
axes[0,0].set_xlabel('£')

axes[0,1].hist(cleaned_df['log_price'], bins=60, color='#3A7D44', edgecolor='white')
axes[0,1].set_title(f'log1p(price) skew={cleaned_df["log_price"].skew():.2f}')
axes[0,1].set_xlabel('log1p(£)')

## Vehicle age distribution
axes[0,2].hist(cleaned_df['vehicle_age'].dropna(), bins=40, color='#D62828', edgecolor='white')
axes[0,2].set_title('Vehicle Age Distribution')
axes[0,2].set_xlabel('Years')

## Mileage vs log price
axes[1,0].scatter(cleaned_df['mileage'], cleaned_df['log_price'],
                  alpha=0.05, s=2, color='#2E86AB')
axes[1,0].set_title('Mileage vs log1p(Price)')
axes[1,0].set_xlabel('Mileage'); axes[1,0].set_ylabel('log1p(£)')

## Vehicle age vs log price
axes[1,1].scatter(cleaned_df['vehicle_age'], cleaned_df['log_price'],
                  alpha=0.05, s=2, color='#3A7D44')
axes[1,1].set_title('Vehicle Age vs log1p(Price)')
axes[1,1].set_xlabel('Years'); axes[1,1].set_ylabel('log1p(£)')

## Average price by vehicle condition
cond_price = cleaned_df.groupby('vehicle_condition')['price'].median().sort_values()
axes[1,2].barh(cond_price.index, cond_price.values, color='#D62828')
axes[1,2].set_title('Median Price by Condition')
axes[1,2].set_xlabel('£')

plt.suptitle('Task 1: EDA — Raw and Cleaned Data', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '01_eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
## 1.5 Subsetting Analysis and Correlation

from scipy.stats import f_oneway

## Pearson correlation matrix
num_cols = ['mileage','vehicle_age','mileage_per_year','age_adjusted_mileage','price']
corr = cleaned_df[num_cols].corr()
plt.figure(figsize=(7,5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True)
plt.title('Pearson Correlation — Numerical Features')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '01_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## ANOVA: categorical features vs price
print('=== ANOVA: Categorical Features vs Price ===')
for cat in ['vehicle_condition','body_type','fuel_type','standard_make']:
    groups = [g['price'].values for _,g in cleaned_df.groupby(cat) if len(g)>1]
    if len(groups)>1:
        f,p = f_oneway(*groups)
        print(f'  {cat}: F={f:.1f}, p={p:.2e}')

## Average price by key categories
print()
for cat in ['vehicle_condition','fuel_type']:
    print(f'=== Average price by {cat} ===')
    display(cleaned_df.groupby(cat)['price'].mean()
            .sort_values(ascending=False).to_frame('mean_price_£').round(0))

In [ ]:
## 1.6 Train/Val/Test Split and Preprocessing Pipeline
## 70/15/15 split — leakage-safe: preprocessor fitted on training data ONLY

EXCLUDED = ['price','log_price','public_reference','advert_date',
            'advert_year','year_of_registration','reg_code',
            'advert_month','depreciation_stage']

## Defensive: don't ask for more rows than the cleaned dataset has
df_work = cleaned_df.sample(
    n=min(SAMPLE_SIZE, len(cleaned_df)),
    random_state=SEED
).copy()
excl     = [c for c in EXCLUDED if c in df_work.columns]
X        = df_work.drop(columns=excl)
y        = np.log1p(df_work['price'])

## Stratified 70/15/15 split by price decile X vehicle_condition
## Why stratify: the price distribution is heavy-tailed and luxury cars (>£50k) are
## rare. A random split can put almost all of them in train, leaving val and test
## artificially easy. Stratifying by price decile keeps each tier proportionally
## represented; crossing with vehicle_condition keeps NEW vs USED proportions stable.
strata = (
    pd.qcut(df_work['price'], q=10, labels=False, duplicates='drop').astype(str)
    + '_' + df_work['vehicle_condition'].astype(str)
)
strata_counts = strata.value_counts()
rare = strata_counts[strata_counts < 2].index
strata = strata.where(~strata.isin(rare), other='_rare')

X_train, X_tmp, y_train, y_tmp, strat_train, strat_tmp = train_test_split(
    X, y, strata, test_size=0.30, random_state=SEED, stratify=strata)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, random_state=SEED, stratify=strat_tmp)

num_features = X_train.select_dtypes(include=['number','bool']).columns.tolist()
cat_features = X_train.select_dtypes(include=['object','category']).columns.tolist()

try:
    ohe = OneHotEncoder(handle_unknown='ignore', min_frequency=50, sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown='ignore', min_frequency=50, sparse=False)

preprocess = ColumnTransformer([
    ('num', Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('sc',  StandardScaler())
    ]), num_features),
    ('cat', Pipeline([
        ('imp', SimpleImputer(strategy='most_frequent')),
        ('ohe', ohe)
    ]), cat_features),
], remainder='drop')

## Tree pipeline — NO StandardScaler (trees are threshold-based not distance-based)
try:
    ohe_tree = OneHotEncoder(handle_unknown='ignore', min_frequency=50, sparse_output=False)
except TypeError:
    ohe_tree = OneHotEncoder(handle_unknown='ignore', min_frequency=50, sparse=False)

tree_prep = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), num_features),
    ('cat', Pipeline([
        ('imp', SimpleImputer(strategy='most_frequent')),
        ('ohe', ohe_tree)
    ]), cat_features),
], remainder='drop')

## Evaluation function — all metrics in original £
def evaluate_model(name, model, X_tr, y_tr, X_v, y_v, X_te, y_te):
    results = {'model': name}
    for split, Xs, ys in [('train',X_tr,y_tr),('val',X_v,y_v),('test',X_te,y_te)]:
        pred = model.predict(Xs)
        y_   = np.expm1(ys); p_ = np.expm1(pred)
        results[f'{split}_MAE_GBP']  = round(mean_absolute_error(y_, p_), 0)
        results[f'{split}_RMSE_GBP'] = round(np.sqrt(mean_squared_error(y_, p_)), 0)
        results[f'{split}_R2']       = round(r2_score(y_, p_), 4)
    ## val_train_MAE_gap = val_MAE - train_MAE: positive means validation is
    ## worse than training (the usual case), which is easier to read on the table.
    results['val_train_MAE_gap'] = round(
        results['val_MAE_GBP'] - results['train_MAE_GBP'], 0)
    return results

def show_metrics(df_m, sort_by='val_MAE_GBP'):
    cols = ['model','train_MAE_GBP','val_MAE_GBP','test_MAE_GBP',
            'train_R2','val_R2','test_R2','val_train_MAE_gap']
    cols = [c for c in cols if c in df_m.columns]
    display(df_m[cols].sort_values(sort_by).reset_index(drop=True))

print(f'Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}')
print(f'Numeric features: {len(num_features)} | Categorical features: {len(cat_features)}')
print()

## Output dataset summary
output_summary = pd.DataFrame({
    'feature': num_features + cat_features + ['price (target)'],
    'type': ['numeric']*len(num_features) + ['categorical']*len(cat_features) + ['target'],
    'engineered': ['yes' if c in ['vehicle_age','mileage_per_year','age_adjusted_mileage',
                   'is_sweet_spot','mileage_is_round','colour_premium',
                   'is_plate_change_month','implicit_repair_risk','is_alternative_fuel']
                   else 'no' for c in num_features + cat_features + ['price (target)']]
})
print('=== OUTPUT DATASET SUMMARY ===')
display(output_summary)
output_summary.to_csv(OUTPUT_DIR / '01_output_dataset_summary.csv', index=False)

print(f'\nModel target: log1p(price) | Reported in original £ via np.expm1')
print(f'reg_code excluded: True | min_frequency=50 OHE: True | Leakage-safe split: True')

---
# 2. Automated Feature Selection (10%)

Automated feature selection reduces the high-dimensional encoded feature space created by one-hot encoding of categorical variables. The aim is to identify the most informative subset of features for predicting `price`, improving interpretability and reducing computational cost.

Four methods from genuinely different paradigms are applied:

| Method | Type | Key idea |
|---|---|---|
| Baseline (all features) | — | Comparison point |
| SelectKBest (f_regression) | Filter, univariate | F-statistic with target |
| SelectFromModel (ExtraTrees) | Model-based | Tree importance threshold |
| RFE with Ridge | Recursive | Iterative elimination after pre-filter |
| SFS forward | Sequential | Greedy forward search on top-50 pool |

In [ ]:
## 2.1 Encode Feature Matrix (leakage-safe: fit on train only)
X_train_enc = preprocess.fit_transform(X_train, y_train)
X_val_enc   = preprocess.transform(X_val)
X_test_enc  = preprocess.transform(X_test)
feat_names  = preprocess.get_feature_names_out()

print(f'Encoded feature matrix (train): {X_train_enc.shape}')
print(f'Raw features: {X_train.shape[1]} → Encoded: {X_train_enc.shape[1]}')

## Ridge evaluation helper — fast, stable on scaled/OHE data
def fit_eval_fs(name, n_feat, Xtr, Xv, Xte):
    m = Ridge(alpha=1.0).fit(Xtr, y_train)
    return {
        'method': name, 'n_features': n_feat,
        'val_MAE_GBP':  round(mean_absolute_error(np.expm1(y_val),  np.expm1(m.predict(Xv))),  0),
        'test_MAE_GBP': round(mean_absolute_error(np.expm1(y_test), np.expm1(m.predict(Xte))), 0),
        'val_R2':  round(r2_score(np.expm1(y_val),  np.expm1(m.predict(Xv))),  4),
        'test_R2': round(r2_score(np.expm1(y_test), np.expm1(m.predict(Xte))), 4),
    }

fs_results = [fit_eval_fs('Baseline: all features', X_train_enc.shape[1],
                           X_train_enc, X_val_enc, X_test_enc)]
print('Baseline done.')
display(pd.DataFrame(fs_results))

In [ ]:
## 2.2 SelectKBest (f_regression) — elbow plot
k_values = [25, 50, 100, 200]
k_values = [k for k in k_values if k < X_train_enc.shape[1]]
kbest_selected = {}

for k in k_values:
    sel = SelectKBest(f_regression, k=k).fit(X_train_enc, y_train)
    fs_results.append(fit_eval_fs(
        f'SelectKBest k={k}', k,
        sel.transform(X_train_enc), sel.transform(X_val_enc), sel.transform(X_test_enc)))
    kbest_selected[k] = feat_names[sel.get_support()]
    print(f'  k={k} done')

## Elbow plot — mirrors Luciano's RFECV curve from lab notebook
kbest_rows = [r for r in fs_results if r['method'].startswith('SelectKBest')]
plt.figure(figsize=(8, 4))
plt.plot([r['n_features'] for r in kbest_rows],
         [r['val_MAE_GBP'] for r in kbest_rows], 'o-', color='#2E86AB', label='Val MAE')
plt.plot([r['n_features'] for r in kbest_rows],
         [r['test_MAE_GBP'] for r in kbest_rows], 's-', color='#3A7D44', label='Test MAE')
plt.xlabel('k (features selected)'); plt.ylabel('MAE (£)')
plt.title('SelectKBest: MAE vs Number of Features (elbow plot)')
plt.legend(); plt.tight_layout()
plt.savefig(OUTPUT_DIR / '02_kbest_elbow.png', dpi=150, bbox_inches='tight')
plt.show()
display(pd.DataFrame(fs_results))

In [ ]:
## 2.3 SelectFromModel (ExtraTrees, median threshold) + RFE with pre-filter

## SelectFromModel — captures non-linear/interaction effects that univariate KBest misses
sfm = SelectFromModel(
    ExtraTreesRegressor(n_estimators=100, max_depth=12, n_jobs=-1, random_state=SEED),
    threshold='median'
).fit(X_train_enc, y_train)
sfm_names = feat_names[sfm.get_support()]
fs_results.append(fit_eval_fs(
    f'SelectFromModel ExtraTrees (median)', sfm.transform(X_train_enc).shape[1],
    sfm.transform(X_train_enc), sfm.transform(X_val_enc), sfm.transform(X_test_enc)))
print(f'SelectFromModel selected: {sfm.transform(X_train_enc).shape[1]} features')

## RFE with pre-filter and ExtraTreesRegressor as the estimator.
## ExtraTrees ranks features by impurity reduction with random thresholds, which is
## robust to correlated OHE columns. RFE on the full OHE space would time out in Colab,
## so SelectKBest pre-filters to a candidate pool first.
## RFE estimator hyperparameters are kept small (n_estimators=80, max_depth=12) and
## step=0.3 so each elimination round halves quickly; otherwise tree-based RFE is slow.
cand_k = min(150, X_train_enc.shape[1])
pref = SelectKBest(f_regression, k=cand_k).fit(X_train_enc, y_train)
Xtr_pref = pref.transform(X_train_enc)
Xv_pref  = pref.transform(X_val_enc)
Xte_pref = pref.transform(X_test_enc)
pref_names = feat_names[pref.get_support()]
rfe_selected = {}

rfe_estimator = ExtraTreesRegressor(
    n_estimators=80, max_depth=12, n_jobs=-1, random_state=SEED)

for n in [30, 50, 100]:
    if n >= cand_k: continue
    rfe = RFE(rfe_estimator, n_features_to_select=n, step=0.3)
    Xtr_r = rfe.fit_transform(Xtr_pref, y_train)
    fs_results.append(fit_eval_fs(
        f'RFE ExtraTrees n={n}', n,
        Xtr_r, rfe.transform(Xv_pref), rfe.transform(Xte_pref)))
    rfe_selected[n] = pref_names[rfe.get_support()]
    print(f'  RFE n={n} done')

display(pd.DataFrame(fs_results))

In [ ]:
## 2.4 Sequential Feature Selection (top-50 pool)
## SFS is included because the assessment spec names it
## Applied on a small pre-filtered pool because it is computationally expensive

sfs_k = min(50, X_train_enc.shape[1])
sfs_pref = SelectKBest(f_regression, k=sfs_k).fit(X_train_enc, y_train)
Xtr_sfs  = sfs_pref.transform(X_train_enc)
Xv_sfs   = sfs_pref.transform(X_val_enc)
Xte_sfs  = sfs_pref.transform(X_test_enc)
sfs_pool = feat_names[sfs_pref.get_support()]

print('Running SFS forward (n=20, cv=3)...')
sfs = SequentialFeatureSelector(
    Ridge(alpha=1.0), n_features_to_select=20,
    direction='forward', scoring='neg_mean_absolute_error', cv=3, n_jobs=-1
)
Xtr_sfs_sel = sfs.fit_transform(Xtr_sfs, y_train)
fs_results.append(fit_eval_fs(
    'SFS forward n=20 (top-50 pool)', 20,
    Xtr_sfs_sel, sfs.transform(Xv_sfs), sfs.transform(Xte_sfs)))
print('SFS done.')
display(pd.DataFrame(fs_results))

In [ ]:
## 2.5 Comparison, Feature Family Summary, and Final Selection

## Full comparison — sorted by val MAE (NOT test MAE)
fs_df = pd.DataFrame(fs_results).sort_values('val_MAE_GBP').reset_index(drop=True)
print('=== FEATURE SELECTION COMPARISON ===')
display(fs_df)
fs_df.to_csv(OUTPUT_DIR / '02_fs_comparison.csv', index=False)

## Bar chart
plt.figure(figsize=(10, 5))
plt.barh(fs_df['method'], fs_df['val_MAE_GBP'], color='#2E86AB')
plt.xlabel('Validation MAE (£)')
plt.title('Automated Feature Selection: Validation MAE Comparison')
plt.gca().invert_yaxis(); plt.tight_layout()
plt.savefig(OUTPUT_DIR / '02_fs_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Feature family summary — which groups survive?
def summarise_families(names):
    families = []
    for f in names:
        clean = f.split('__',1)[1] if '__' in f else f
        families.append(clean.split('_')[0])
    return pd.Series(families).value_counts().reset_index().rename(
        columns={'index':'family', 0:'count'})

print('\nFeature families retained by SelectFromModel:')
display(summarise_families(sfm_names).head(8))

best_fs = fs_df.iloc[0]
print(f"\nBest method: {best_fs['method']} | n={best_fs['n_features']} | "
      f"val MAE £{best_fs['val_MAE_GBP']:,.0f} | test MAE £{best_fs['test_MAE_GBP']:,.0f}")
print('Full feature set retained for Tasks 3-6 — tree ensembles handle high-dim OHE efficiently')

---
# 3. Tree Ensembles (10%)

Individual decision trees are unstable — small changes in training data produce very different splits. Random Forests reduce variance by averaging many trees trained on different data samples and feature subsets (parallel variance reduction). HistGradientBoosting builds trees sequentially, with later trees correcting errors from earlier ones (sequential error correction).

**Hyperparameter tuning:** `RandomizedSearchCV` with `neg_mean_absolute_error` scoring. `val_train_MAE_gap` (val MAE - train MAE) is monitored to detect overfitting.

In [ ]:
## 3.1 Define Tree Preprocessing and Evaluation

## Tree-specific preprocessing — NO StandardScaler
## Trees split on thresholds, not distances — scaling adds no value
try:
    ohe_t3 = OneHotEncoder(handle_unknown='ignore', min_frequency=50, sparse_output=False)
except TypeError:
    ohe_t3 = OneHotEncoder(handle_unknown='ignore', min_frequency=50, sparse=False)

tree_prep3 = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), num_features),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                      ('ohe', ohe_t3)]), cat_features),
], remainder='drop')

tree_results = []
print('Tree preprocessing defined (no StandardScaler — trees are threshold-based)')

In [ ]:
## 3.2 Random Forest — Baseline and Tuned

## Baseline RF — untuned
rf_base = Pipeline([('prep', tree_prep3),
                    ('model', RandomForestRegressor(
                        n_estimators=150, random_state=SEED, n_jobs=-1))])
print('Fitting RF baseline...', end=' ')
rf_base.fit(X_train, y_train)
tree_results.append(evaluate_model('RF baseline', rf_base,
    X_train, y_train, X_val, y_val, X_test, y_test))
print(f'val MAE: £{tree_results[-1]["val_MAE_GBP"]:,.0f}')

## Tuned RF — RandomizedSearchCV
rf_search = RandomizedSearchCV(
    Pipeline([('prep', tree_prep3),
              ('model', RandomForestRegressor(random_state=SEED, n_jobs=-1))]),
    param_distributions={
        'model__n_estimators':     [150, 250, 400],
        'model__max_depth':        [10, 18, 25, None],
        'model__min_samples_leaf': [1, 3, 5, 10],
        'model__max_features':     ['sqrt', 0.5, 0.8],
    },
    n_iter=10, scoring='neg_mean_absolute_error',
    cv=3, random_state=SEED, n_jobs=-1, verbose=1
)
print('Tuning RF (n_iter=10, cv=3)...')
rf_search.fit(X_train, y_train)
rf_tuned = rf_search.best_estimator_
print('Best params:', rf_search.best_params_)
tree_results.append(evaluate_model('RF tuned', rf_tuned,
    X_train, y_train, X_val, y_val, X_test, y_test))
show_metrics(pd.DataFrame(tree_results))

In [ ]:
## 3.3 HistGradientBoosting — Baseline and Tuned

## HGB baseline
hgb_base = Pipeline([('prep', tree_prep3),
                     ('model', HistGradientBoostingRegressor(
                         max_iter=200, learning_rate=0.08,
                         max_leaf_nodes=31, random_state=SEED))])
print('Fitting HGB baseline...', end=' ')
hgb_base.fit(X_train, y_train)
tree_results.append(evaluate_model('HGB baseline', hgb_base,
    X_train, y_train, X_val, y_val, X_test, y_test))
print(f'val MAE: £{tree_results[-1]["val_MAE_GBP"]:,.0f}')

## Tuned HGB
hgb_search = RandomizedSearchCV(
    Pipeline([('prep', tree_prep3),
              ('model', HistGradientBoostingRegressor(random_state=SEED))]),
    param_distributions={
        'model__max_iter':          [150, 250, 400],
        'model__learning_rate':     [0.03, 0.06, 0.1],
        'model__max_leaf_nodes':    [15, 31, 63],
        'model__min_samples_leaf':  [20, 50, 100],
        'model__l2_regularization': [0.0, 0.1, 1.0],
    },
    n_iter=12, scoring='neg_mean_absolute_error',
    cv=3, random_state=SEED, n_jobs=-1, verbose=1
)
print('Tuning HGB (n_iter=12, cv=3)...')
hgb_search.fit(X_train, y_train)
hgb_tuned = hgb_search.best_estimator_
print('Best params:', hgb_search.best_params_)
tree_results.append(evaluate_model('HGB tuned', hgb_tuned,
    X_train, y_train, X_val, y_val, X_test, y_test))
show_metrics(pd.DataFrame(tree_results))

In [ ]:
## 3.4 Comparison, Visualisation and Error Analysis

tree_df = pd.DataFrame(tree_results).sort_values('val_MAE_GBP').reset_index(drop=True)
print('=== TREE ENSEMBLE RESULTS ===')
show_metrics(tree_df)
tree_df.to_csv(OUTPUT_DIR / '03_tree_metrics.csv', index=False)

## Bar chart
plt.figure(figsize=(9, 4))
plt.barh(tree_df['model'], tree_df['val_MAE_GBP'], color='#2E86AB')
plt.xlabel('Validation MAE (£)')
plt.title('Task 3: Tree Ensemble Models — Validation MAE')
plt.gca().invert_yaxis(); plt.tight_layout()
plt.savefig(OUTPUT_DIR / '03_tree_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

best_t3_name = tree_df.iloc[0]['model']
best_t3 = {'RF baseline':rf_base,'RF tuned':rf_tuned,
            'HGB baseline':hgb_base,'HGB tuned':hgb_tuned}[best_t3_name]
print(f'Best model: {best_t3_name}')

In [ ]:
## 3.5 Actual vs Predicted, Residuals, and Error by Price Band

pred_test   = np.expm1(best_t3.predict(X_test))
y_true_test = np.expm1(y_test)
residuals   = y_true_test - pred_test

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

## Actual vs Predicted
axes[0].scatter(y_true_test, pred_test, s=4, alpha=0.2, color='#2E86AB')
lim = float(np.percentile(y_true_test, 99))
axes[0].plot([0,lim],[0,lim], color='red', lw=1.5, label='Perfect')
axes[0].set_xlabel('Actual (£)'); axes[0].set_ylabel('Predicted (£)')
axes[0].set_title(f'Actual vs Predicted — {best_t3_name}'); axes[0].legend()

## Residuals
axes[1].scatter(pred_test, residuals, s=4, alpha=0.2, color='#3A7D44')
axes[1].axhline(0, color='red', lw=1.5)
axes[1].set_xlabel('Predicted (£)'); axes[1].set_ylabel('Residual (£)')
axes[1].set_title('Residuals vs Predicted')

## Error by price band — model reliability across market segments
err_df = pd.DataFrame({'actual':y_true_test,'abs_error':np.abs(residuals)})
err_df['band'] = pd.cut(err_df['actual'],
    bins=[0,5000,10000,20000,35000,50000,np.inf],
    labels=['<£5k','£5-10k','£10-20k','£20-35k','£35-50k','£50k+'])
sns.boxplot(data=err_df, x='band', y='abs_error', ax=axes[2], color='#2E86AB')
axes[2].set_title('Error by Price Band\n(model reliability across market)')
axes[2].set_xlabel('Price Band'); axes[2].set_ylabel('Absolute Error (£)')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '03_actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

## Generalisation gap table
print('=== GENERALISATION GAP (val MAE - train MAE) ===')
for _,r in tree_df.iterrows():
    print(f"{r['model']:22s} | gap: £{r['val_train_MAE_gap']:>5,.0f} | "
          f"val R²: {r['val_R2']} | test R²: {r['test_R2']}")

## Median error by price band
print('\nMedian absolute error by price band:')
print(err_df.groupby('band')['abs_error'].median().round(0))

---
# 4. Ensemble of Tree Ensembles (10%)

Three combination strategies are tested. Each is evaluated on validation MAE — test set used only as final confirmation.

| Strategy | Rationale |
|---|---|
| Voting (equal) | Tests whether model diversity helps |
| Voting (weighted) | Best Task 3 model gets weight 2 — data-driven |
| Stacking (Ridge meta) | Learns optimal weighting from base model outputs |

Ridge is used as the stacking meta-learner — using another tree would rediscover the same partitions and double-count the signal.

In [ ]:
## 4.1 Define Base Learners

try:
    ohe_t4 = OneHotEncoder(handle_unknown='ignore', min_frequency=50, sparse_output=False)
except TypeError:
    ohe_t4 = OneHotEncoder(handle_unknown='ignore', min_frequency=50, sparse=False)

tree_prep4 = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), num_features),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                      ('ohe', ohe_t4)]), cat_features),
], remainder='drop')

rf4 = Pipeline([('prep', tree_prep4),
                ('model', RandomForestRegressor(
                    n_estimators=250, max_depth=25, min_samples_leaf=3,
                    max_features='sqrt', random_state=SEED, n_jobs=-1))])

et4 = Pipeline([('prep', tree_prep4),
                ('model', ExtraTreesRegressor(
                    n_estimators=250, max_depth=25, min_samples_leaf=3,
                    max_features='sqrt', random_state=SEED, n_jobs=-1))])

hgb4 = Pipeline([('prep', tree_prep4),
                 ('model', HistGradientBoostingRegressor(
                     max_iter=250, learning_rate=0.06, max_leaf_nodes=31,
                     min_samples_leaf=50, l2_regularization=0.1, random_state=SEED))])

ens_results = []
for name, mdl in [('Random Forest',rf4),('Extra Trees',et4),('HistGradientBoosting',hgb4)]:
    print(f'Fitting {name}...')
    mdl.fit(X_train, y_train)
    ens_results.append(evaluate_model(name, mdl,
        X_train, y_train, X_val, y_val, X_test, y_test))
    print(f'  val MAE: £{ens_results[-1]["val_MAE_GBP"]:,.0f}')
show_metrics(pd.DataFrame(ens_results))

In [ ]:
## 4.2 Voting Ensembles (equal and weighted)

## Equal weights
print('Fitting Voting (equal weights)...')
voting_eq = VotingRegressor([('rf',rf4),('et',et4),('hgb',hgb4)], n_jobs=-1)
voting_eq.fit(X_train, y_train)
ens_results.append(evaluate_model('Voting (equal)', voting_eq,
    X_train, y_train, X_val, y_val, X_test, y_test))
print(f'  val MAE: £{ens_results[-1]["val_MAE_GBP"]:,.0f}')

## Weighted — HGB gets weight 2 (best individual from Task 3 typically)
print('Fitting Voting (weighted, HGB×2)...')
voting_w = VotingRegressor([('rf',rf4),('et',et4),('hgb',hgb4)],
                           weights=[1,1,2], n_jobs=-1)
voting_w.fit(X_train, y_train)
ens_results.append(evaluate_model('Voting (weighted HGB×2)', voting_w,
    X_train, y_train, X_val, y_val, X_test, y_test))
print(f'  val MAE: £{ens_results[-1]["val_MAE_GBP"]:,.0f}')
show_metrics(pd.DataFrame(ens_results))

In [ ]:
## 4.3 Stacking with Ridge Meta-Learner
## cv=3 for speed — cv=5 would be more robust with more compute
## Ridge chosen so meta-learner doesn't rediscover same tree partitions

print('Fitting Stacking (RF+ET+HGB → Ridge, cv=3)...')
stacking = StackingRegressor(
    estimators=[('rf',rf4),('et',et4),('hgb',hgb4)],
    final_estimator=Ridge(alpha=1.0),
    passthrough=False, cv=3, n_jobs=-1
)
stacking.fit(X_train, y_train)
ens_results.append(evaluate_model('Stacking → Ridge', stacking,
    X_train, y_train, X_val, y_val, X_test, y_test))
print(f'  val MAE: £{ens_results[-1]["val_MAE_GBP"]:,.0f}')

In [ ]:
## 4.4 Full Comparison and Final Selection

ens_df = pd.DataFrame(ens_results).sort_values('val_MAE_GBP').reset_index(drop=True)
print('=== ENSEMBLE RESULTS ===')
show_metrics(ens_df)
ens_df.to_csv(OUTPUT_DIR / '04_ensemble_metrics.csv', index=False)

plt.figure(figsize=(10, 5))
plt.barh(ens_df['model'], ens_df['val_MAE_GBP'], color='#2E86AB')
plt.xlabel('Validation MAE (£)')
plt.title('Task 4: Ensemble of Tree Ensembles — Validation MAE')
plt.gca().invert_yaxis(); plt.tight_layout()
plt.savefig(OUTPUT_DIR / '04_ensemble_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

best_ens = ens_df.iloc[0]
print(f"Best ensemble: {best_ens['model']}")
print(f"val MAE: £{best_ens['val_MAE_GBP']:,.0f} | test MAE: £{best_ens['test_MAE_GBP']:,.0f}")
print(f"val-train gap: £{best_ens['val_train_MAE_gap']:,.0f}")

---
# 5. Feature Importance (10%)

**Task 5 answers: which features matter?**
**Task 6 answers: how do those features affect predictions?**

Two complementary methods are used on the best individual model from Task 3:
- **Permutation importance** — measures MAE increase (in £) when each feature is shuffled. Model-agnostic. Corrects impurity-based bias toward high-cardinality features.
- **SHAP (TreeExplainer)** — game-theoretic attributions summing to the prediction. Computed on 1,000 validation rows to keep Colab stable.

In [ ]:
## 5.1 Select Model and Explanation Samples

model_to_explain = best_t3
print(f'Model for explanation: {type(model_to_explain.named_steps["model"]).__name__}')

## Permutation: 3,000 rows | SHAP: 1,000 rows
X_perm = X_val.sample(min(3000, len(X_val)), random_state=SEED)
y_perm = y_val.loc[X_perm.index]
X_shap_raw = X_val.sample(min(1000, len(X_val)), random_state=SEED)
y_shap     = y_val.loc[X_shap_raw.index]

print(f'Permutation sample: {X_perm.shape} | SHAP sample: {X_shap_raw.shape}')

In [ ]:
## 5.2 Permutation Importance — GBP MAE scorer
## Importance = increase in MAE (£) when feature is shuffled
## More interpretable than log-scale importance

def neg_mae_gbp(y_true_log, y_pred_log):
    return -mean_absolute_error(np.expm1(y_true_log), np.expm1(y_pred_log))

gbp_scorer = make_scorer(neg_mae_gbp, greater_is_better=True)

print('Computing permutation importance (n_repeats=5)...')
perm = permutation_importance(
    model_to_explain, X_perm, y_perm,
    scoring=gbp_scorer, n_repeats=5, random_state=SEED, n_jobs=-1
)

perm_df = pd.DataFrame({
    'feature': X_perm.columns,
    'importance_mean_GBP': perm.importances_mean,
    'importance_std':      perm.importances_std,
}).sort_values('importance_mean_GBP', ascending=False).reset_index(drop=True)

print('Top 15 features by permutation importance (£ MAE increase):')
display(perm_df.head(15))
perm_df.to_csv(OUTPUT_DIR / '05_permutation_importance.csv', index=False)

## Plot with error bars
plot_pi = perm_df.head(15).sort_values('importance_mean_GBP')
plt.figure(figsize=(10, 7))
plt.barh(plot_pi['feature'], plot_pi['importance_mean_GBP'],
         xerr=plot_pi['importance_std'], color='#2E86AB', ecolor='#555', capsize=3)
plt.xlabel('Increase in val MAE (£) when feature shuffled')
plt.title('Permutation Importance — Top 15\n(5 repeats, error bars = std)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '05_permutation_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
## 5.3 SHAP Values — TreeExplainer

fitted_prep = model_to_explain.named_steps['prep']
fitted_est  = model_to_explain.named_steps['model']

X_shap_enc = fitted_prep.transform(X_shap_raw)
if hasattr(X_shap_enc, 'toarray'): X_shap_enc = X_shap_enc.toarray()
enc_names  = fitted_prep.get_feature_names_out()
X_shap_df  = pd.DataFrame(X_shap_enc, columns=enc_names, index=X_shap_raw.index)

if HAS_SHAP:
    print('Computing SHAP values (TreeExplainer)...')
    explainer = shap.TreeExplainer(fitted_est)
    shap_vals = explainer.shap_values(X_shap_df)
    if isinstance(shap_vals, list): shap_vals = shap_vals[0]
    print(f'SHAP values shape: {np.array(shap_vals).shape}')
else:
    print('SHAP not available — install with: !pip install shap')

In [ ]:
## 5.4 Grouped SHAP + Rank Comparison

def map_to_original(enc_name, cat_feats):
    if enc_name.startswith('num__'): return enc_name.replace('num__','',1)
    if enc_name.startswith('cat__'):
        rem = enc_name.replace('cat__','',1)
        for col in sorted(cat_feats, key=len, reverse=True):
            if rem == col or rem.startswith(col+'_'): return col
    return enc_name

if HAS_SHAP:
    raw_names = [map_to_original(f, cat_features) for f in enc_names]
    shap_imp = pd.DataFrame({
        'encoded_feature': enc_names,
        'raw_feature': raw_names,
        'mean_abs_shap': np.abs(shap_vals).mean(axis=0)
    })
    shap_grouped = (shap_imp.groupby('raw_feature', as_index=False)['mean_abs_shap'].sum()
                   .sort_values('mean_abs_shap', ascending=False).reset_index(drop=True))

    ## Beeswarm plot
    plt.figure(figsize=(10,7))
    shap.summary_plot(shap_vals, X_shap_df, feature_names=enc_names,
                      max_display=15, show=False)
    plt.title('SHAP Beeswarm — Encoded Features')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / '05_shap_beeswarm.png', dpi=150, bbox_inches='tight')
    plt.show()

    ## Grouped bar
    top12 = shap_grouped.head(12).sort_values('mean_abs_shap')
    plt.figure(figsize=(9,6))
    plt.barh(top12['raw_feature'], top12['mean_abs_shap'], color='#2E86AB')
    plt.xlabel('Mean absolute SHAP value')
    plt.title('Grouped SHAP Importance by Original Feature')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / '05_shap_grouped.png', dpi=150, bbox_inches='tight')
    plt.show()

    ## Rank comparison
    comp = perm_df[['feature','importance_mean_GBP']].merge(
        shap_grouped.rename(columns={'raw_feature':'feature'}), on='feature', how='outer')
    comp['perm_rank'] = comp['importance_mean_GBP'].rank(ascending=False)
    comp['shap_rank'] = comp['mean_abs_shap'].rank(ascending=False)
    print('=== PERMUTATION vs SHAP RANK COMPARISON ===')
    display(comp.sort_values('perm_rank', na_position='last').head(15))

In [ ]:
## 5.5 Local SHAP — Waterfall for One Advert

if HAS_SHAP:
    try:
        ev = explainer.expected_value
        if isinstance(ev, (list, np.ndarray)): ev = np.array(ev).ravel()[0]
        explanation = shap.Explanation(
            values=shap_vals[0], base_values=ev,
            data=X_shap_df.iloc[0].values, feature_names=enc_names)
        shap.plots.waterfall(explanation, max_display=15)
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / '05_shap_waterfall.png', dpi=150, bbox_inches='tight')
    except Exception as e:
        print(f'Waterfall skipped: {e}')

    actual = float(np.expm1(y_shap.iloc[0]))
    pred   = float(np.expm1(model_to_explain.predict(X_shap_raw.iloc[[0]])[0]))
    print(f'Advert 0 — Actual: £{actual:,.0f} | Predicted: £{pred:,.0f} | '
          f'Error: £{abs(actual-pred):,.0f}')

---
# 6. SHAP/PDP Model Explanations (10%)

Task 5 answered **which features matter**. Task 6 answers **how those features affect predictions** — globally, on average, and for individual adverts.

| Tool | Purpose |
|---|---|
| SHAP dependence plots | How one feature's contribution varies across its value range |
| One-way PDP | Average predicted log-price as a feature changes |
| ICE curves | Individual variation around the average PDP |
| Two-way PDP | Interaction between `vehicle_age` and `mileage` |

In [ ]:
## 6.1 SHAP Dependence Plots

X_pdp = X_val.sample(min(3000, len(X_val)), random_state=SEED)

if HAS_SHAP:
    target_feats = [f for f in ['num__vehicle_age','num__mileage',
                                 'num__age_adjusted_mileage']
                    if f in enc_names]
    print(f'SHAP dependence features: {target_feats}')
    for feat in target_feats[:2]:
        shap.dependence_plot(feat, shap_vals, X_shap_df,
                             feature_names=enc_names, show=False)
        plt.title(f'SHAP Dependence — {feat}')
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / f'06_shap_dep_{feat.replace("__","_")}.png',
                    dpi=150, bbox_inches='tight')
        plt.show()

In [ ]:
## 6.2 One-Way PDP

pdp_features = [f for f in ['vehicle_age','mileage','age_adjusted_mileage',
                              'year_of_registration']
                if f in X_pdp.columns]
print(f'PDP features: {pdp_features[:3]}')

for feat in pdp_features[:3]:
    fig, ax = plt.subplots(figsize=(7, 5))
    PartialDependenceDisplay.from_estimator(
        model_to_explain, X_pdp, features=[feat],
        kind='average', ax=ax)
    ax.set_title(f'PDP: {feat}')
    ax.set_ylabel('Predicted log1p(price)')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'06_pdp_{feat}.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
## 6.3 ICE Plots — Individual Variation Around PDP
## Luciano's exact style: kind='both', red PDP line, subsample=100

for feat in pdp_features[:2]:
    fig, ax = plt.subplots(figsize=(8, 5))
    PartialDependenceDisplay.from_estimator(
        model_to_explain, X_pdp, features=[feat],
        kind='both', subsample=100, random_state=SEED, ax=ax,
        pd_line_kw={'color': 'red', 'linewidth': 2}
    )
    ax.set_title(f'PDP + ICE: {feat}\n(red = average PDP | grey = individual ICE)')
    ax.set_ylabel('Predicted log1p(price)')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'06_ice_{feat}.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
## 6.4 Two-Way PDP (vehicle_age × mileage interaction)

pairs = []
if 'vehicle_age' in X_pdp.columns and 'mileage' in X_pdp.columns:
    pairs = [('vehicle_age','mileage')]
elif 'age_adjusted_mileage' in X_pdp.columns and 'mileage' in X_pdp.columns:
    pairs = [('age_adjusted_mileage','mileage')]

if pairs:
    fig, ax = plt.subplots(figsize=(8,6))
    PartialDependenceDisplay.from_estimator(
        model_to_explain, X_pdp, features=pairs, kind='average', ax=ax)
    ax.set_title('Two-Way PDP: Vehicle Age × Mileage Interaction')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / '06_twoway_pdp.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
## 6.5 SHAP/PDP Interpretation Summary Table

## NOTE: this table is intentionally NEUTRAL.
## Interpretations must be written in the report AFTER inspecting the actual
## SHAP / PDP / ICE outputs produced above - never copy hard-coded findings,
## otherwise the report can claim something the plots do not actually show.
interpretation_summary = pd.DataFrame({
    'tool': [
        'SHAP global beeswarm',
        'SHAP grouped bar',
        'SHAP dependence',
        'SHAP waterfall',
        'One-way PDP',
        'ICE curves',
        'Two-way PDP'
    ],
    'what_it_explains': [
        'Encoded feature magnitude and direction',
        'Original feature importance after grouping OHE columns',
        'How one feature contribution varies across its value range',
        'Why one individual advert received its predicted price',
        'Average model response as one feature changes',
        'Whether the average effect holds for individual adverts',
        'Interaction between two important numerical features'
    ],
    'interpretation_to_write_after_outputs': [
        'Use actual top SHAP features from the plot',
        'Use actual grouped SHAP ranking',
        'Describe actual direction shown in the dependence plot',
        'Explain this advert only, not the whole market',
        'Describe the observed PDP trend',
        'State whether ICE curves are stable or varied',
        'Describe whether age and mileage effects compound'
    ]
})
display(interpretation_summary)
interpretation_summary.to_csv(OUTPUT_DIR / '06_interpretation_summary.csv', index=False)

---
# 7. Dimensionality Reduction — Linear: PCA (10%)

PCA is applied to the **numerical and boolean features only** (not the full OHE matrix). PCA on sparse OHE columns rediscovers high-frequency category indicators — the scree plot becomes uninformative and the PCs are uninterpretable. Restricting to numerics+flags produces interpretable components.

A Ridge baseline on PCA components is compared against the full-feature baseline to quantify the information cost of dimensionality reduction.

In [ ]:
## 7.1 Fit PCA on Numerical and Boolean Features

PCA_COLS = [c for c in num_features if c in X_train.columns]
print(f'PCA features (numerics only): {PCA_COLS}')

## Fit imputer+scaler on training data only
num_pipe_pca = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('sc',  StandardScaler())
])
Xn_train = num_pipe_pca.fit_transform(X_train[PCA_COLS])
Xn_val   = num_pipe_pca.transform(X_val[PCA_COLS])
Xn_test  = num_pipe_pca.transform(X_test[PCA_COLS])

pca = PCA(random_state=SEED).fit(Xn_train)
print(f'PCA components: {pca.n_components_} (over {len(PCA_COLS)} features)')
print(f'Top-1 variance: {pca.explained_variance_ratio_[0]*100:.1f}%')

In [ ]:
## 7.2 Scree Plot and Cumulative Variance

evr = pca.explained_variance_ratio_
cum = np.cumsum(evr)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(range(1, len(evr)+1), evr*100, color='#2E86AB', alpha=0.85, label='Per-PC variance')
axes[0].plot(range(1, len(cum)+1), cum*100, 'o-', color='#D62828', label='Cumulative')
for thresh in [90, 95]:
    axes[0].axhline(thresh, color='grey', lw=0.8, ls=':')
    axes[0].text(len(evr)*0.95, thresh+1, f'{thresh}%', color='grey', fontsize=8, ha='right')
axes[0].set_title('Scree Plot — Numerical Features')
axes[0].set_xlabel('PC index'); axes[0].set_ylabel('Variance explained (%)')
axes[0].legend()

## PC loadings heatmap
loadings = pd.DataFrame(pca.components_, columns=PCA_COLS,
                        index=[f'PC{i+1}' for i in range(len(evr))])
sns.heatmap(loadings.iloc[:min(6,len(evr))], annot=True, fmt='.2f',
            cmap='coolwarm', center=0, ax=axes[1])
axes[1].set_title('PC Loadings Heatmap (first 6 PCs)')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '07_pca_scree.png', dpi=150, bbox_inches='tight')
plt.show()

k90 = int(np.argmax(cum >= 0.90)) + 1
k95 = int(np.argmax(cum >= 0.95)) + 1
print(f'Components for 90% variance: {k90} | 95% variance: {k95}')

In [ ]:
## 7.3 2D PCA Scatter — Coloured by log_price

Xn_train_2d = pca.transform(Xn_train)[:, :2]

plt.figure(figsize=(8, 6))
sc = plt.scatter(Xn_train_2d[:,0], Xn_train_2d[:,1],
                 c=y_train, cmap='viridis', alpha=0.3, s=3)
plt.colorbar(sc, label='log1p(price)')
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.title('PCA 2D Embedding — Coloured by log1p(Price)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '07_pca_2d.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
## 7.4 Downstream: Ridge on PCA Components vs Full Feature Baseline

pca_results = []
for k in [2, k90, k95, len(evr)]:
    k = min(k, len(evr))
    Xtr_k = pca.transform(Xn_train)[:, :k]
    Xv_k  = pca.transform(Xn_val)[:, :k]
    Xte_k = pca.transform(Xn_test)[:, :k]
    m = Ridge(alpha=1.0).fit(Xtr_k, y_train)
    pca_results.append({
        'model': f'Ridge on {k} PCs',
        'n_components': k,
        'val_MAE_GBP':  round(mean_absolute_error(np.expm1(y_val),  np.expm1(m.predict(Xv_k))),  0),
        'test_MAE_GBP': round(mean_absolute_error(np.expm1(y_test), np.expm1(m.predict(Xte_k))), 0),
        'val_R2':       round(r2_score(np.expm1(y_val),  np.expm1(m.predict(Xv_k))),  4),
    })

pca_df = pd.DataFrame(pca_results)
print('=== PCA + Ridge vs Full-Feature Baseline ===')
display(pca_df)
print()
print('Note: Ridge on PCA components uses numerics only — much weaker than')
print('full-feature model because categorical signal (standard_make, model) is discarded.')
pca_df.to_csv(OUTPUT_DIR / '07_pca_results.csv', index=False)

---
# 8. Dimensionality Reduction — Non-Linear: Isomap and t-SNE (10%)

**Isomap** fits a geodesic distance matrix — O(n²) memory. Applied to a 5,000-row subsample only. `n_neighbors` is swept to show how the local-radius parameter shapes the manifold.

**t-SNE** is used for visualisation only — it does not produce a reusable transform for new data, so it cannot be evaluated downstream. Isomap is used for the downstream comparison.

In [ ]:
## 8.1 Isomap — n_neighbors Sweep

## 5K subsample — Isomap is O(n²) memory
N_ISO = 5000
iso_idx  = np.random.choice(len(Xn_train), size=min(N_ISO, len(Xn_train)), replace=False)
Xn_iso   = Xn_train[iso_idx]
y_iso    = y_train.iloc[iso_idx].values

iso_results = []
for n_nb in [5, 15, 50]:
    print(f'Fitting Isomap n_neighbors={n_nb}...', end=' ')
    iso = Isomap(n_neighbors=n_nb, n_components=2)
    X_iso_2d = iso.fit_transform(Xn_iso)
    iso_results.append({'n_neighbors': n_nb, 'embedding': X_iso_2d, 'isomap': iso})
    print('done')

In [ ]:
## 8.2 Isomap 2D Embeddings — Coloured by log_price

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, res in zip(axes, iso_results):
    sc = ax.scatter(res['embedding'][:,0], res['embedding'][:,1],
                    c=y_iso, cmap='viridis', alpha=0.3, s=3)
    ax.set_title(f'Isomap n_neighbors={res["n_neighbors"]}')
    ax.set_xlabel('Component 1'); ax.set_ylabel('Component 2')
    plt.colorbar(sc, ax=ax, label='log1p(price)')
plt.suptitle('Isomap 2D Embeddings (5K subsample) — Coloured by log1p(Price)', fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '08_isomap_embeddings.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
## 8.3 t-SNE Visualisation (2D, visualisation only)
## t-SNE does not produce a reusable transform — used for exploration only

from sklearn.manifold import TSNE
print('Fitting t-SNE (perplexity=30, 2D, on Isomap subsample)...')
tsne = TSNE(n_components=2, perplexity=30, random_state=SEED, n_jobs=-1)
X_tsne = tsne.fit_transform(Xn_iso)

plt.figure(figsize=(8, 6))
sc = plt.scatter(X_tsne[:,0], X_tsne[:,1],
                 c=y_iso, cmap='viridis', alpha=0.4, s=3)
plt.colorbar(sc, label='log1p(price)')
plt.xlabel('t-SNE 1'); plt.ylabel('t-SNE 2')
plt.title('t-SNE 2D Embedding — Coloured by log1p(Price)\n(visualisation only — no reusable transform)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '08_tsne.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
## 8.4 Downstream: Ridge on Isomap Components

## Use n_neighbors=15 (cleanest manifold from the sweep)
best_iso = iso_results[1]['isomap']  ## n_neighbors=15

iso_downstream = []
for k in [2, 3, 4]:
    iso_k = Isomap(n_neighbors=15, n_components=k).fit(Xn_iso)
    Xtr_i = iso_k.transform(Xn_train)
    Xv_i  = iso_k.transform(Xn_val)
    Xte_i = iso_k.transform(Xn_test)
    m = Ridge(alpha=1.0).fit(Xtr_i, y_train)
    iso_downstream.append({
        'model': f'Ridge on {k} Isomap components (n_nb=15)',
        'n_components': k,
        'val_MAE_GBP':  round(mean_absolute_error(np.expm1(y_val),  np.expm1(m.predict(Xv_i))),  0),
        'test_MAE_GBP': round(mean_absolute_error(np.expm1(y_test), np.expm1(m.predict(Xte_i))), 0),
        'val_R2':       round(r2_score(np.expm1(y_val),  np.expm1(m.predict(Xv_i))),  4),
    })

iso_df = pd.DataFrame(iso_downstream)
print('=== Isomap + Ridge Downstream Results ===')
display(iso_df)
print()
print('Comparison — PCA vs Isomap vs Full-feature baseline:')
print(f'  Ridge on PCA (k={k95}):    val MAE £{pca_df.iloc[-2]["val_MAE_GBP"]:,.0f}')
print(f'  Ridge on Isomap (k=4):     val MAE £{iso_df.iloc[-1]["val_MAE_GBP"]:,.0f}')
print(f'  Full feature tree model:   val MAE ~£2,900-3,200 (Tasks 3-4)')
print('Both DR methods use numerics only — categorical signal dominates this problem.')
iso_df.to_csv(OUTPUT_DIR / '08_isomap_results.csv', index=False)

---
# 9. Polynomial Regression (10%)

Polynomial regression with Ridge regularisation is included as a smooth-model comparison to the tree ensembles. This is directly relevant because recent tabular-regression benchmarking work reports that smooth-basis models can be statistically competitive with tree ensembles on accuracy while exhibiting tighter generalisation gaps among CPU-viable models (Gerber & Lloyd, 2026, arXiv:2602.22422).

**Design:** Polynomial expansion is applied to **numerical features only**. Expanding the full OHE matrix (90+ columns) to degree 2 would produce 4,000+ features and the result would be uninterpretable. The categorical encodings are concatenated unmodified.

In [ ]:
## 9.1 Polynomial + Ridge Pipeline

def make_poly_prep(degree):
    """Polynomial expansion on numerics only, OHE on categoricals unchanged."""
    if degree <= 1:
        num_path = Pipeline([
            ('imp', SimpleImputer(strategy='median')),
            ('sc',  StandardScaler())
        ])
    else:
        num_path = Pipeline([
            ('imp',  SimpleImputer(strategy='median')),
            ('sc',   StandardScaler()),
            ('poly', PolynomialFeatures(degree=degree,
                                        interaction_only=False,
                                        include_bias=False))
        ])
    try:
        ohe_p = OneHotEncoder(handle_unknown='ignore', min_frequency=50, sparse_output=False)
    except TypeError:
        ohe_p = OneHotEncoder(handle_unknown='ignore', min_frequency=50, sparse=False)

    return ColumnTransformer([
        ('num', num_path, num_features),
        ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                          ('ohe', ohe_p)]), cat_features),
    ], remainder='drop')

def fit_poly_ridge(degree, alpha=1.0):
    pipe = Pipeline([('prep', make_poly_prep(degree)),
                     ('reg',  Ridge(alpha=alpha))])
    pipe.fit(X_train, y_train)
    return pipe

print('Polynomial-Ridge factory ready.')
print('Numeric expansion only — full OHE expansion would produce 4,000+ features')

In [ ]:
## 9.2 Validation Curve — Degree × Alpha Grid

degrees = [1, 2, 3]
alphas  = [1.0, 10.0, 100.0, 1000.0]
vc_rows = []

for d in degrees:
    for a in alphas:
        pipe = fit_poly_ridge(d, a)
        tr_mae = mean_absolute_error(np.expm1(y_train), np.expm1(pipe.predict(X_train)))
        va_mae = mean_absolute_error(np.expm1(y_val),   np.expm1(pipe.predict(X_val)))
        vc_rows.append({
            'degree': d, 'alpha': a,
            'train_MAE_GBP': round(tr_mae, 0),
            'val_MAE_GBP':   round(va_mae, 0),
            'train_val_gap': round(tr_mae - va_mae, 0)
        })
        print(f'  deg={d} alpha={a}: val MAE £{va_mae:,.0f}')

vc_df = pd.DataFrame(vc_rows)
print('\n=== VALIDATION CURVE RESULTS ===')
display(vc_df)

In [ ]:
## 9.3 Validation Curve Plot

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for a in alphas:
    sub = vc_df[vc_df['alpha']==a].sort_values('degree')
    axes[0].plot(sub['degree'], sub['train_MAE_GBP'], 'o--', alpha=0.5, label=f'train α={a:g}')
    axes[0].plot(sub['degree'], sub['val_MAE_GBP'],   'o-',              label=f'val α={a:g}')
axes[0].set_title('Validation Curve — MAE vs Polynomial Degree')
axes[0].set_xlabel('Degree'); axes[0].set_ylabel('MAE (£)')
axes[0].set_xticks(degrees); axes[0].legend(fontsize=7, ncol=2)

## Best config
best_idx = vc_df['val_MAE_GBP'].idxmin()
best_d, best_a = int(vc_df.loc[best_idx,'degree']), float(vc_df.loc[best_idx,'alpha'])
print(f'Best config: degree={best_d}, alpha={best_a}')
best_pipe = fit_poly_ridge(best_d, best_a)
y_val_pred = np.expm1(best_pipe.predict(X_val))
axes[1].scatter(np.expm1(y_val), y_val_pred, s=4, alpha=0.2, color='#2E86AB')
lim = float(np.percentile(np.expm1(y_val), 99))
axes[1].plot([0,lim],[0,lim],'r--',lw=1.5)
axes[1].set_xlabel('Actual (£)'); axes[1].set_ylabel('Predicted (£)')
axes[1].set_title(f'Pred vs Actual — deg={best_d}, alpha={best_a}')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '09_poly_validation_curve.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
## 9.4 Polynomial vs Linear vs Best Tree — Final Comparison

linear_pipe = fit_poly_ridge(1, 1.0)
linear_mae  = mean_absolute_error(np.expm1(y_val), np.expm1(linear_pipe.predict(X_val)))
poly_mae    = mean_absolute_error(np.expm1(y_val), np.expm1(best_pipe.predict(X_val)))
linear_r2   = r2_score(np.expm1(y_val), np.expm1(linear_pipe.predict(X_val)))
poly_r2     = r2_score(np.expm1(y_val), np.expm1(best_pipe.predict(X_val)))

## Test set evaluation — single pass
linear_test_mae = mean_absolute_error(np.expm1(y_test), np.expm1(linear_pipe.predict(X_test)))
poly_test_mae   = mean_absolute_error(np.expm1(y_test), np.expm1(best_pipe.predict(X_test)))

poly_comparison = pd.DataFrame([
    {'model':'Linear Ridge (deg=1)',     'val_MAE_GBP':round(linear_mae,0), 'test_MAE_GBP':round(linear_test_mae,0), 'val_R2':round(linear_r2,4)},
    {'model':f'Poly Ridge (deg={best_d}, α={best_a})','val_MAE_GBP':round(poly_mae,0),'test_MAE_GBP':round(poly_test_mae,0),'val_R2':round(poly_r2,4)},
    {'model':'Best tree (from Task 3)',  'val_MAE_GBP':tree_df.iloc[0]['val_MAE_GBP'],
     'test_MAE_GBP':tree_df.iloc[0]['test_MAE_GBP'], 'val_R2':tree_df.iloc[0]['val_R2']},
])
print('=== POLYNOMIAL vs LINEAR vs TREE ENSEMBLE ===')
display(poly_comparison)
poly_comparison.to_csv(OUTPUT_DIR / '09_poly_comparison.csv', index=False)
print()
print('Smooth polynomial models exhibit tighter generalisation gaps than tree ensembles')
print('but lower absolute accuracy — consistent with Gerber & Lloyd (2026, arXiv:2602.22422).')

---
# 10. Clustering for Feature Engineering (10%)

K-Means is used as a **feature factory** rather than a final model. Cluster-derived features are appended to the design matrix and evaluated by whether they improve downstream Ridge regression performance.

Both the **cluster ID (one-hot)** and **distance-to-each-centroid vector** are exposed as features. The cluster ID encodes hard segment membership; distance-to-centroid encodes soft membership. Both are tested together, and their value is judged by validation MAE — not assumed in advance.

In [ ]:
## 10.1 Elbow and Silhouette — Choose Optimal K

## Two matrices are built here:
##   (1) X_train_enc_km / val / test  - the FULL encoded feature matrix (numeric + OHE).
##       This is used as the downstream Ridge BASELINE in 10.5, and is also the matrix
##       we augment with cluster features in 10.3.
##   (2) X_train_km / val / test      - a numeric-only USAGE matrix used to FIT K-Means.
##       Clustering on the full encoded matrix tends to be driven by make/model/category
##       OHE dummies; clustering on numeric usage features keeps the segments interpretable
##       in terms of vehicle_age / mileage / repair-risk, which is what we discuss in the report.

X_train_enc_km = preprocess.fit_transform(X_train, y_train)
X_val_enc_km   = preprocess.transform(X_val)
X_test_enc_km  = preprocess.transform(X_test)

cluster_features = [
    'vehicle_age',
    'mileage',
    'mileage_per_year',
    'age_adjusted_mileage',
    'implicit_repair_risk',
    'is_alternative_fuel',
    'is_plate_change_month',
]
cluster_features = [c for c in cluster_features if c in X_train.columns]
print(f'K-Means clustering features (numeric usage features): {cluster_features}')

km_pipe = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('sc',  StandardScaler())
])
X_train_km = km_pipe.fit_transform(X_train[cluster_features])
X_val_km   = km_pipe.transform(X_val[cluster_features])
X_test_km  = km_pipe.transform(X_test[cluster_features])

## 5K subsample for elbow — keep the sweep fast and stable
n_sub  = min(5000, X_train_km.shape[0])
km_idx = np.random.choice(X_train_km.shape[0], size=n_sub, replace=False)
Xkm_sub = X_train_km[km_idx]

k_range  = range(2, 13)
inertias = []
silhs    = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    labels = km.fit_predict(Xkm_sub)
    inertias.append(km.inertia_)
    silhs.append(silhouette_score(Xkm_sub, labels, sample_size=2000, random_state=SEED))
    print(f'  k={k}: inertia={km.inertia_:.0f} silhouette={silhs[-1]:.4f}')

In [ ]:
## 10.2 Elbow + Silhouette Plots

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(list(k_range), inertias, 'o-', color='#2E86AB')
axes[0].set_xlabel('Number of clusters k')
axes[0].set_ylabel('Inertia (within-cluster sum of squares)')
axes[0].set_title('Elbow Plot — Choose k at the bend')

axes[1].plot(list(k_range), silhs, 's-', color='#D62828')
axes[1].set_xlabel('Number of clusters k')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score — Higher = better-separated clusters')

best_k_sil = list(k_range)[np.argmax(silhs)]
axes[1].axvline(best_k_sil, color='green', linestyle='--', label=f'Best k={best_k_sil}')
axes[1].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '10_kmeans_elbow.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Best k by silhouette: {best_k_sil}')

In [ ]:
## 10.3 Fit Final K-Means and Create Feature Matrix

BEST_K = best_k_sil

## K-Means is fitted on the numeric usage matrix (X_train_km), so segments are
## interpretable in terms of vehicle_age / mileage / repair-risk rather than OHE dummies.
km_final = KMeans(n_clusters=BEST_K, random_state=SEED, n_init=10)
km_final.fit(X_train_km)

## Cluster IDs
train_cluster_ids = km_final.predict(X_train_km)
val_cluster_ids   = km_final.predict(X_val_km)
test_cluster_ids  = km_final.predict(X_test_km)

## Distances to each centroid (soft membership)
train_dists = km_final.transform(X_train_km)
val_dists   = km_final.transform(X_val_km)
test_dists  = km_final.transform(X_test_km)

## One-hot encode cluster IDs
try:
    ohe_km = OneHotEncoder(sparse_output=False)
except TypeError:
    ohe_km = OneHotEncoder(sparse=False)

train_km_ohe = ohe_km.fit_transform(train_cluster_ids.reshape(-1,1))
val_km_ohe   = ohe_km.transform(val_cluster_ids.reshape(-1,1))
test_km_ohe  = ohe_km.transform(test_cluster_ids.reshape(-1,1))

## Augmented matrices: original features + cluster OHE + distances
X_train_aug = np.hstack([X_train_enc_km, train_km_ohe, train_dists])
X_val_aug   = np.hstack([X_val_enc_km,   val_km_ohe,   val_dists])
X_test_aug  = np.hstack([X_test_enc_km,  test_km_ohe,  test_dists])

print(f'Original encoded shape: {X_train_enc_km.shape}')
print(f'Augmented shape:        {X_train_aug.shape} (+{BEST_K*2} cluster features)')

In [ ]:
## 10.4 Cluster Profile — Interpret What Clusters Represent

cluster_profile = X_train[['mileage','vehicle_age','mileage_per_year',
                             'age_adjusted_mileage']].copy()
cluster_profile['cluster'] = train_cluster_ids
cluster_profile['log_price'] = y_train.values
cluster_profile['price_GBP'] = np.expm1(y_train.values)

profile_summary = cluster_profile.groupby('cluster').agg({
    'price_GBP':           ['mean','median','count'],
    'vehicle_age':         'mean',
    'mileage':             'mean',
    'age_adjusted_mileage':'mean',
}).round(0)
print('=== CLUSTER PROFILE ===')
display(profile_summary)
print()
print('Note: price was NOT used to fit the K-Means clusters — it is shown here only')
print('to interpret the clusters AFTER the fact. The cluster profiles indicate whether')
print('the unsupervised vehicle segments are associated with different price levels.')
print('Whether the segments help prediction is decided by validation MAE in 10.5.')

In [ ]:
## 10.5 Downstream: Does Clustering Improve Prediction?

m_orig = Ridge(alpha=1.0).fit(X_train_enc_km, y_train)
m_aug  = Ridge(alpha=1.0).fit(X_train_aug,    y_train)

orig_val_mae = mean_absolute_error(np.expm1(y_val), np.expm1(m_orig.predict(X_val_enc_km)))
aug_val_mae  = mean_absolute_error(np.expm1(y_val), np.expm1(m_aug.predict(X_val_aug)))
orig_test_mae= mean_absolute_error(np.expm1(y_test),np.expm1(m_orig.predict(X_test_enc_km)))
aug_test_mae = mean_absolute_error(np.expm1(y_test),np.expm1(m_aug.predict(X_test_aug)))

km_comparison = pd.DataFrame([
    {'model':'Ridge (no clustering)',
     'val_MAE_GBP':round(orig_val_mae,0), 'test_MAE_GBP':round(orig_test_mae,0),
     'val_R2':round(r2_score(np.expm1(y_val), np.expm1(m_orig.predict(X_val_enc_km))),4)},
    {f'model':f'Ridge + KMeans (k={BEST_K}, OHE + distances)',
     'val_MAE_GBP':round(aug_val_mae,0),  'test_MAE_GBP':round(aug_test_mae,0),
     'val_R2':round(r2_score(np.expm1(y_val), np.expm1(m_aug.predict(X_val_aug))),4)},
])
print('=== CLUSTERING FEATURE ENGINEERING RESULTS ===')
display(km_comparison)
km_comparison.to_csv(OUTPUT_DIR / '10_kmeans_results.csv', index=False)

improvement = orig_val_mae - aug_val_mae
print(f'\nVal MAE change from cluster features: £{improvement:,.0f}')
print(f'(positive = cluster features help; negative = they hurt; near zero = no effect)')
print(f'Both cluster ID (one-hot) and distance-to-centroid vectors are tested because')
print(f'they represent hard and soft segment membership. Their value is judged by val MAE.')

---
# Final Summary — Cross-Task Results

All models evaluated. Test set used once per model after validation-based selection.

In [ ]:
## Final cross-task comparison table

final_summary = {
    'Task 2 — Best Feature Selection':
        f"{fs_df.iloc[0]['method']} | val MAE £{fs_df.iloc[0]['val_MAE_GBP']:,.0f}",
    'Task 3 — Best Tree Ensemble':
        f"{tree_df.iloc[0]['model']} | val MAE £{tree_df.iloc[0]['val_MAE_GBP']:,.0f} | test MAE £{tree_df.iloc[0]['test_MAE_GBP']:,.0f} | R² {tree_df.iloc[0]['test_R2']}",
    'Task 4 — Best Ensemble of Ensembles':
        f"{ens_df.iloc[0]['model']} | val MAE £{ens_df.iloc[0]['val_MAE_GBP']:,.0f} | test MAE £{ens_df.iloc[0]['test_MAE_GBP']:,.0f}",
    'Task 7 — PCA':
        f'k95={k95} components | val MAE (Ridge) £{pca_df.iloc[-2]["val_MAE_GBP"]:,.0f}',
    'Task 9 — Best Polynomial':
        f'deg={best_d} alpha={best_a} | val MAE £{round(poly_mae,0):,.0f} | test MAE £{round(poly_test_mae,0):,.0f}',
    'Task 10 — KMeans Feature Engineering':
        f'k={BEST_K} | improvement £{round(improvement,0):,.0f} on val MAE',
}

for task, result in final_summary.items():
    print(f'{task}:')
    print(f'  {result}')
    print()

print('All outputs saved to aml_outputs/ directory')
print('Notebook complete — ready for report writing')